# Impulse — Reporting Pipeline Demo

The **Impulse Framework** is a Python library that enables
automotive and industrial engineers to process, aggregate,
and analyze petabytes of time-series measurement data on
Databricks — without requiring Apache Spark expertise.

It provides **TSAL** (Time Series Analytics Language),
a Pythonic expression language for defining signals,
events, and aggregations.

**What this notebook builds:**
A complete reporting pipeline — RPM histograms,
RPM-vs-speed heatmaps, and per-distance-bin statistics
across 3 test drives — persisted as a Gold-layer
star schema with an auto-generated AI/BI dashboard.

**Requirements:**
- Databricks workspace with **Unity Catalog** access
- **Serverless** compute (or DBR 14+)

## Architecture

![Impulse architecture](../docs/img/architecture.svg)

Impulse sits between a governed silver layer and a gold-layer star schema in Unity Catalog and provides three components:

- **TSAL (Time Series Analytics Language)** — a declarative Python DSL for expressing signals, events, and aggregations in natural Python, without requiring Spark expertise.
- **Query Engine** — pluggable and distributed; compiles TSAL expressions into Spark execution plans and adapts to any silver-layer layout via interchangeable solvers.
- **Aggregations** — domain-aware physical aggregations, including duration- and distance-weighted 1D/2D histograms and event-scoped statistics.

# 1. Setup

In [0]:
%pip install pydantic>=2.0 scipy -q
dbutils.library.restartPython()

### Configure target location

Fill in **Catalog**, **Schema**, and **Table Prefix**
in the widgets above, then run the next cells.

In [0]:
dbutils.widgets.text("catalog", "", "Catalog")
dbutils.widgets.text("schema", "", "Schema")
dbutils.widgets.text(
    "table_prefix", "", "Table Prefix"
)
dbutils.widgets.dropdown("drop_created_tables", "false", ["true", "false"], "Drop Created Tables")

In [0]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import sys, os

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("schema")
TABLE_PREFIX = dbutils.widgets.get("table_prefix")

if not CATALOG or not SCHEMA:
    raise ValueError(
        "Please set Catalog and Schema "
        "widgets above before running."
    )

nb_path = (
    dbutils.notebook.entry_point.getDbutils()
    .notebook().getContext().notebookPath().get()
)
DEMOS_DIR = (
    "/Workspace"
    + "/".join(nb_path.split("/")[:-1])
)
REPO_ROOT = (
    "/Workspace"
    + "/".join(nb_path.split("/")[:-2])
)
sys.path.insert(0, os.path.join(REPO_ROOT, "src"))

pfx = f"{CATALOG}.{SCHEMA}.{TABLE_PREFIX}"
print(f"Target: {pfx}_*")

### Load demo data into Silver layer

Impulse reads from a **Silver layer**:
- **Container** = one measurement recording
  (e.g., one test drive)
- **Channel** = one sensor signal within a container
  (e.g., Engine RPM), stored as raw
  `(timestamp, value)` samples — the framework
  automatically converts these to intervals on the fly

In [0]:
import pandas as pd

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS "
    f"{CATALOG}.{SCHEMA}"
)

csv_dir = os.path.join(DEMOS_DIR, "data", "reporting")

SILVER = [
    "container_metrics", "container_tags",
    "channel_metrics", "channel_tags",
    "channels",
]
for t in SILVER:
    pdf = pd.read_csv(f"{csv_dir}/{t}.csv")
    spark.createDataFrame(pdf).write.mode(
        "overwrite"
    ).saveAsTable(f"{pfx}_{t}")

print(f"Loaded {len(SILVER)} tables")

In [0]:
display(spark.read.table(f"{pfx}_container_metrics"))

In [0]:
display(spark.read.table(f"{pfx}_channels").limit(10))

## Pipeline Overview

The diagram below shows the end-to-end flow:
**Silver Layer** inputs and **user-defined** channels
and aggregations feed into Impulse Framework,
which produces a **Gold Layer** star schema.

![MDA Pipeline Overview](images/mda_pipeline_overview.png)

# 2. Initialize the Report

The `Report` orchestrator takes a config specifying:
- **`source`** — Silver layer tables
- **`unity_sink`** — Gold layer output
- **`query_engine.solver`** — `DeltaSolver` for
  parallel per-container execution
- **`query_engine.data_type`** — `RAW` for raw
  timestamp data (auto-converted to intervals)
- **`measurement_dimensions`** — container metadata
  to carry into Gold layer

In [0]:
from databricks.sdk import WorkspaceClient
from impulse_reporting.core.report import Report
from impulse_reporting.core.page import Page
from impulse_reporting.aggregations.histogram import HistogramDuration
from impulse_reporting.aggregations.histogram2d import Histogram2DDuration
from impulse_reporting.aggregations.stats_aggregator import StatsAggregator
from impulse_reporting.events.basic_event import BasicEvent
from impulse_reporting.events.container_event import ContainerEvent
import pyspark.sql.functions as F

ws = WorkspaceClient()

config = {
    "source": {
        "container_metrics_table": f"{pfx}_container_metrics",
        "channel_metrics_table": f"{pfx}_channel_metrics",
        "channels_uri": f"{pfx}_channels",
        "container_tags_table": f"{pfx}_container_tags",
        "channel_tags_table": f"{pfx}_channel_tags",
    },
    "unity_sink": {
        "catalog": CATALOG,
        "schema": SCHEMA,
        "table_prefix": TABLE_PREFIX,
    },
    "query_engine": {
        "solver": "DeltaSolver",
        "data_type": "RAW",
    },
    "measurement_dimensions": [
        "container_id", "vehicle_key",
        "start_ts", "stop_ts",
    ],
}

report = Report(
    name="cuj1_demo", spark=spark, config=config,
    workspace_client=ws
)
db = report.get_db()
print("Report initialized")

# 3. Select Physical Channels

Channels are selected by **metadata tags** —
no column names, no SQL, no joins.
These are **lazy expressions**: no data is read yet.

In [0]:
eng_rpm = db.query.channel(
    channel_name="Engine RPM",
    brand="Seat", model="Leon",
)
veh_spd = db.query.channel(
    channel_name="Vehicle Speed Sensor",
    brand="Seat", model="Leon",
)
amb_air_temp = db.query.channel(
    channel_name="Ambient Air Temperature",
    brand="Seat", model="Leon",
)
intake_air_temp = db.query.channel(
    channel_name="Intake Air Temperature",
    brand="Seat", model="Leon",
)

# 4. Define Virtual Signals & Events

**TSAL** uses Python operators to build lazy
expression trees — no Spark knowledge needed.

**Virtual signals** derive from physical channels.
**Events** are time windows where a condition holds.

In [0]:
avg_temp = (amb_air_temp + intake_air_temp) / 2
distance_km = (
    veh_spd.resample(1e6).cumtrapz() / 3600 / 1e6
)

rpm_band = (eng_rpm > 2000) & (eng_rpm < 5000)
every_10km = (
    (distance_km % 10)
    .intervals_between_falling_edges()
)

# 5. Register Events

- **BasicEvent** — from a TSAL boolean expression
- **ContainerEvent** — spans the entire recording

In [0]:
rpm_event = BasicEvent(
    name="rpm_event",
    expr=rpm_band,
    desc="Engine RPM between 2000 and 5000",
    required_channels=["Engine RPM"],
)
report.add_event(rpm_event)

container_event = ContainerEvent(
    name="container_event",
    desc="Full measurement recording",
)
report.add_event(container_event)

distance_event = BasicEvent(
    name="distance_event",
    expr=every_10km,
    desc="Every 10 km driven",
)
report.add_event(distance_event)

# 6. Define Aggregations

- **Histogram** — 1D duration-weighted distribution
- **Histogram2D** — 2D heatmap of two signals
- **StatisticsAggregator** — min, median, mean, max per event

In [ ]:
page = Page(page_number=1)
report.add_page(page)

sigs = [eng_rpm, veh_spd, amb_air_temp,
        intake_air_temp, avg_temp]
names = ["Engine RPM", "Vehicle Speed",
         "Ambient Air Temp", "Intake Air Temp",
         "Avg Temp"]
aggs = ["min", "median", "mean", "max"]

page.add_aggregation(HistogramDuration(
    name="rpm_histogram", base_expr=eng_rpm,
    bins=[float(i) for i in range(0, 5000, 250)],
    event=rpm_event,
    desc="RPM distribution within RPM events",
    channel_name="Engine RPM",
    bins_unit="RPM", values_unit="s",
))
page.add_aggregation(HistogramDuration(
    name="speed_histogram", base_expr=veh_spd,
    bins=[float(i) for i in range(0, 200, 10)],
    event=rpm_event,
    desc="Speed distribution within RPM events",
    channel_name="Vehicle Speed",
    bins_unit="km/h", values_unit="s",
))
page.add_aggregation(Histogram2DDuration(
    name="rpm_speed_heatmap",
    x_expr=eng_rpm, y_expr=veh_spd,
    x_bins=[float(i) for i in range(2000, 5000, 250)],
    y_bins=[float(i) for i in range(0, 200, 10)],
    event=rpm_event, desc="RPM vs Speed heatmap",
    x_channel_name="Engine RPM",
    y_channel_name="Vehicle Speed",
    x_bins_unit="RPM", y_bins_unit="km/h",
    values_unit="s",
))

page.add_aggregation(StatsAggregator(
    name="rpm_event_stats", input_expressions=sigs,
    channel_names=names, statistics=aggs,
    event=rpm_event,
    desc="Statistics within RPM events",
))
page.add_aggregation(StatsAggregator(
    name="container_stats", input_expressions=sigs,
    channel_names=names, statistics=aggs,
    event=container_event,
    desc="Statistics for full measurement",
))
page.add_aggregation(StatsAggregator(
    name="distance_stats",
    input_expressions=sigs + [distance_km],
    channel_names=names + ["Distance"],
    statistics=aggs,
    event=distance_event,
    desc="Statistics per 10 km distance bin",
))
print(f"{len(page.aggregations)} aggregations added")

# 7. Compute & Persist

- `determine_report()` — parallel execution
- `persist_results()` — writes star schema

In [0]:
report.determine_report()
report.persist_results()
print(f"Report persisted to {pfx}_*")

# 8. Create AI/BI Dashboard

Creates a **Lakeview dashboard** from the
Gold layer tables.

**Note:** To view the dashboard, you need a
**SQL Serverless Warehouse** selected in the
dashboard settings.

In [0]:
import json, uuid
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
host = spark.conf.get(
    "spark.databricks.workspaceUrl"
)

q = {
    "rpm_histogram": (
        f"SELECT bin_name, lower_bound,"
        f" SUM(hist_value / 1e6) AS duration_s"
        f" FROM {pfx}_histogram_fact"
        f" JOIN {pfx}_histogram_dimension"
        f" USING (visual_id)"
        f" WHERE name = 'rpm_histogram'"
        f" GROUP BY bin_name, lower_bound, bin_ID"
        f" ORDER BY lower_bound"
    ),
    "rpm_speed_heatmap": (
        f"SELECT x_bin_name AS rpm_band,"
        f" y_bin_name AS speed_band,"
        f" SUM(hist_value / 1e6) AS duration_s"
        f" FROM {pfx}_histogram2d_fact"
        f" JOIN {pfx}_histogram2d_dimension"
        f" USING (visual_id)"
        f" WHERE name = 'rpm_speed_heatmap'"
        f" GROUP BY x_bin_name, y_bin_name,"
        f" x_lower_bound, y_lower_bound"
        f" ORDER BY x_lower_bound, y_lower_bound"
    ),
    "container_stats": (
        f"SELECT container_id, channel_name,"
        f" aggregation_label, statistic_value"
        f" FROM {pfx}_stats_aggregator_fact"
        f" JOIN {pfx}_stats_aggregator_dimension"
        f" USING (visual_id)"
        f" WHERE name = 'container_stats'"
        f" ORDER BY container_id,"
        f" channel_name, aggregation_label"
    ),
}

_id = lambda: uuid.uuid4().hex[:8]

ds = [
    {
        "name": n,
        "displayName": n.replace("_", " ").title(),
        "queryLines": [v],
    }
    for n, v in q.items()
]

C = {"type": "categorical"}
Q = {"type": "quantitative"}

lo = [
    {
        "widget": {
            "name": _id(),
            "queries": [{
                "name": "main_query",
                "query": {
                    "datasetName": "rpm_histogram",
                    "fields": [
                        {"name": "bin_name",
                         "expression": "`bin_name`"},
                        {"name": "duration_s",
                         "expression": "`duration_s`"},
                    ],
                    "disaggregated": True,
                },
            }],
            "spec": {
                "version": 3,
                "widgetType": "bar",
                "encodings": {
                    "x": {
                        "fieldName": "bin_name",
                        "scale": C,
                        "displayName": "RPM Band",
                    },
                    "y": {
                        "fieldName": "duration_s",
                        "scale": Q,
                        "displayName": "Duration (s)",
                    },
                    "label": {"show": True},
                },
                "frame": {
                    "showTitle": True,
                    "title": "RPM Histogram",
                },
            },
        },
        "position": {
            "x": 0, "y": 0,
            "width": 3, "height": 6,
        },
    },
    {
        "widget": {
            "name": _id(),
            "queries": [{
                "name": "main_query",
                "query": {
                    "datasetName": "rpm_speed_heatmap",
                    "fields": [
                        {"name": "rpm_band",
                         "expression": "`rpm_band`"},
                        {"name": "speed_band",
                         "expression":
                             "`speed_band`"},
                        {"name": "duration_s",
                         "expression": "`duration_s`"},
                    ],
                    "disaggregated": True,
                },
            }],
            "spec": {
                "version": 3,
                "widgetType": "heatmap",
                "encodings": {
                    "x": {
                        "fieldName": "speed_band",
                        "scale": C,
                        "displayName": "Speed Band",
                    },
                    "y": {
                        "fieldName": "rpm_band",
                        "scale": C,
                        "displayName": "RPM Band",
                    },
                    "color": {
                        "fieldName": "duration_s",
                        "scale": Q,
                        "displayName": "Duration (s)",
                    },
                },
                "frame": {
                    "showTitle": True,
                    "title":
                        "RPM vs Speed Heatmap",
                },
            },
        },
        "position": {
            "x": 3, "y": 0,
            "width": 3, "height": 6,
        },
    },
    {
        "widget": {
            "name": _id(),
            "queries": [{
                "name": "main_query",
                "query": {
                    "datasetName": "container_stats",
                    "fields": [
                        {"name": "container_id",
                         "expression":
                             "`container_id`"},
                        {"name": "channel_name",
                         "expression":
                             "`channel_name`"},
                        {"name": "aggregation_label",
                         "expression":
                             "`aggregation_label`"},
                        {"name": "statistic_value",
                         "expression": "`statistic_value`"},
                    ],
                    "disaggregated": True,
                },
            }],
            "spec": {
                "version": 2,
                "widgetType": "table",
                "encodings": {
                    "columns": [
                        {"fieldName": "container_id",
                         "displayName": "Container"},
                        {"fieldName": "channel_name",
                         "displayName": "Signal"},
                        {"fieldName":
                             "aggregation_label",
                         "displayName": "Aggregation"},
                        {"fieldName": "statistic_value",
                         "displayName": "Value"},
                    ],
                },
                "frame": {
                    "showTitle": True,
                    "title": "Container Statistics",
                },
            },
        },
        "position": {
            "x": 0, "y": 6,
            "width": 6, "height": 6,
        },
    },
]

ser = json.dumps({
    "datasets": ds,
    "pages": [{
        "name": _id(),
        "displayName": "Impulse Report",
        "pageType": "PAGE_TYPE_CANVAS",
        "layout": lo,
    }],
})

dash_name = f"Impulse-Report-{TABLE_PREFIX}"

# Find existing dashboard to overwrite
existing_id = None
resp = w.api_client.do(
    "GET", "/api/2.0/lakeview/dashboards"
)
for d in resp.get("dashboards", []):
    if d.get("display_name") == dash_name:
        existing_id = d["dashboard_id"]
        break

if existing_id:
    w.api_client.do(
        "PATCH",
        f"/api/2.0/lakeview/dashboards"
        f"/{existing_id}",
        body={"serialized_dashboard": ser},
    )
    did = existing_id
    print("Dashboard updated")
else:
    resp = w.api_client.do(
        "POST", "/api/2.0/lakeview/dashboards",
        body={
            "display_name": dash_name,
            "serialized_dashboard": ser,
        },
    )
    did = resp["dashboard_id"]
    print("Dashboard created")

url = f"https://{host}/sql/dashboardsv3/{did}"
print(f"Dashboard: {url}")
displayHTML(
    f'<a href="{url}" target="_blank">'
    f"Open Dashboard</a>"
)

# 9. Schedule as a Workflow

Creates a paused job running this notebook
daily at 06:00.

In [0]:
from databricks.sdk.service.jobs import (
    Task, NotebookTask,
    CronSchedule, PauseStatus,
)

job = w.jobs.create(
    name=f"Impulse-Report-{TABLE_PREFIX}",
    tasks=[Task(
        task_key="report",
        notebook_task=NotebookTask(
            notebook_path=("/Workspace" + nb_path),
            base_parameters={
                "catalog": CATALOG,
                "schema": SCHEMA,
                "table_prefix": TABLE_PREFIX,
            },
        ),
    )],
    schedule=CronSchedule(
        quartz_cron_expression="0 0 6 * * ?",
        timezone_id="Europe/Berlin",
        pause_status=PauseStatus.PAUSED,
    ),
)
jurl = f"https://{host}/#job/{job.job_id}"
print(f"Job (paused): {jurl}")
displayHTML(f'<a href="{jurl}" target="_blank">Open Job</a>')

# 10. Cleanup

Drop all tables created by this demo.

In [0]:
if dbutils.widgets.get("drop_created_tables") == "true":
    tables_to_drop = [
        *[f"{pfx}_{t}" for t in SILVER],
        f"{pfx}_histogram_fact",
        f"{pfx}_histogram_dimension",
        f"{pfx}_histogram2d_fact",
        f"{pfx}_histogram2d_dimension",
        f"{pfx}_stats_aggregator_fact",
        f"{pfx}_stats_aggregator_dimension",
        f"{pfx}_event_instance_fact",
        f"{pfx}_event_dimension",
        f"{pfx}_measurement_dimension",
    ]
    for t in tables_to_drop:
        spark.sql(f"DROP TABLE IF EXISTS {t}")
    print(f"Dropped {len(tables_to_drop)} tables")